# AEGIS AI — Kill-Feed Detector Training (YOLO26 Nano) 🎯

**Ye notebook kya karta hai:** Wahi kaam jo pehle notebook (RF-DETR) me tha — kill-feed box pehchanne wala model — lekin **Ultralytics YOLO** se. YOLO nano models **CPU par bhi tez** chalte hain, isliye ye *alternative* hai: jiska ONNX seedha AEGIS ke `VISION_MODEL_PATH` me lagta hai, bina kisi extra post-processing ke.

**Dataset:** 100,000 frames, **10 chunk zips** me (~800MB each). Notebook saare chunks **khud** unzip karke ek dataset bana lega. `train/` + `val/` ke saath **`test/` split** bhi hai (500-1000 clean frames, ek bilkul alag tournament se) — final report card usi par banega.

**Steps (bilkul pehle notebook jaise):**
1. **GPU on karo:** Settings → Accelerator → **GPU T4**.
2. **Dataset jodho:** *+ Add data* → apna `aegis-killfeed-dataset`.
3. **Cell 3** me `DATASET_SLUG` me apna Kaggle username likho.
4. **Run** (▶▶) dabao — 100k images × 50 epochs me **kai ghante** lagenge.

**YOLO26 ke baare me imaandaar note:** YOLO26 Ultralytics ka asli latest model hai (2025 release). Notebook pehle `yolo26n.pt` load karne ki koshish karega; agar Kaggle ke ultralytics version me ye na mile to **khud-ba-khud `yolov8n.pt` par switch** kar jayega aur aapko bata dega. Dono se kaam chal jayega.


In [ ]:
# Cell 1 — Ultralytics install karo
!pip install -q ultralytics onnx
print("Install complete ✔")


## Step 1 — GPU check 🔌

In [ ]:
# Cell 2 — GPU hai ya nahi?
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU naam:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU NAHI mila! Settings → Accelerator → GPU T4 select karo.")


## Step 2 — Dataset jodho 📦

`DATASET_SLUG` me apna Kaggle **username** likho (example: `"nishant07/aegis-killfeed-dataset"`).


In [ ]:
# Cell 3 — Apna dataset slug yahan likho (sirf username badalna hai)
DATASET_SLUG = "username/aegis-killfeed-dataset"  # ← "username" ki jagah apna Kaggle username likho

# Saare chunk zips unzip karo + ek saaf YOLO dataset banao (chunk-by-chunk, disk bachane ke liye)
import zipfile, shutil, random, yaml
from pathlib import Path

DS = Path("/kaggle/working/aegis_ds")  # final dataset: images/train, images/val, labels/train, labels/val
if DS.exists():
    shutil.rmtree(DS)

zips = sorted(Path("/kaggle/input").rglob("*.zip"))
assert zips, "Koi zip chunk nahi mila — dataset attach karo (+ Add data → Your Datasets)"
print(f"{len(zips)} zip chunks mile")

CLASS_NAMES = ["killfeed"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
random.seed(42)
counts = {"train": 0, "val": 0, "test": 0}

for ci, z in enumerate(zips):
    tmp = Path(f"/tmp/chunk_{ci:02d}")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(tmp)
    if ci == 0:  # pehle chunk me dataset.yaml mila to class names wahi se lo
        for y in tmp.rglob("dataset.yaml"):
            n = yaml.safe_load(open(y)).get("names", ["killfeed"])
            CLASS_NAMES = n if isinstance(n, list) else [n[k] for k in sorted(n)]
            break
    lbl_by_stem = {}
    for t in tmp.rglob("*.txt"):
        lbl_by_stem.setdefault(t.stem, t)
    imgs = sorted(p for p in tmp.rglob("*") if p.suffix.lower() in IMG_EXTS)
    for img in imgs:
        parts = [p.lower() for p in img.parts]
        if "train" in parts:
            split = "train"
        elif "test" in parts:
            split = "test"  # alag tournament ke clean frames — training me KABHI use nahi honge
        elif "val" in parts or "valid" in parts:
            split = "val"
        else:
            split = "val" if random.random() < 0.10 else "train"  # 90/10 split
        dest_img = DS / "images" / split / f"chunk{ci:02d}_{img.name}"
        dest_img.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, dest_img)
        lbl = lbl_by_stem.get(img.stem)
        if lbl is not None:
            dest_lbl = DS / "labels" / split / (dest_img.stem + ".txt")
            dest_lbl.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(lbl, dest_lbl)
        counts[split] += 1
    shutil.rmtree(tmp)
    print(f"✔ {z.name}: {len(imgs)} images")

print("Classes:", CLASS_NAMES)
yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val",
                "test": "images/test", "names": CLASS_NAMES},
               open(DS / "dataset.yaml", "w"))
print(f"Saaf dataset taiyaar ✔  train: {counts['train']}, val: {counts['val']}, test: {counts['test']}  → {DS}")
if counts["test"] == 0:
    print("⚠️ WARNING: test/ split nahi mila! 'Final report card' (asli duniya ka andaza) nahi ban payega.")
    print("   Dataset me 'test' naam ke folder me 500-1000 clean frames (ek alag tournament se) jodho.")
else:
    print(f"✔ test/ me {counts['test']} clean frames — ye training me kabhi use nahi honge (final report card).")


## Step 3 — Model load karo (YOLO26 → fallback YOLOv8) 🧠

In [ ]:
# Cell 4 — Pehle yolo26n.pt, na mile to yolov8n.pt (automatic fallback)
from ultralytics import YOLO

MODEL_TAG = "yolo26n"
try:
    model = YOLO("yolo26n.pt")  # Kaggle se download hoga (pehli baar internet chahiye)
    print("✔ YOLO26-nano load ho gaya (latest model).")
except Exception as e:
    print("yolo26n.pt nahi mil paya, isliye fallback lag raha hai:")
    print("   wajah:", str(e)[:150])
    model = YOLO("yolov8n.pt")
    MODEL_TAG = "yolov8n"
    print("✔ YOLOv8-nano load ho gaya (fallback). Chinta mat karo — kill-feed jaise simple kaam me")
    print("  dono ka result lagbhag ek jaisa aata hai; farak sirf speed/accuracy me thoda sa hai.")
print("Model tag:", MODEL_TAG)


## Step 4 — Training 🏋️ (sabse lamba step)

**50 epochs**, image size 640, batch 16.

⏳ **Time (imaandaar note):** 100,000 images × 50 epochs T4 GPU par **kai ghante** lagenge. Exact time pehle se nahi bataya ja sakta; Run dabane ke baad jo ETA dikhe, use dekho.
- **Kaggle session limit:** GPU session **max ~9-12 ghante** chalta hai (Kaggle docs: 12 ghante; GPU tips page: 9 ghante) — uske baad session band ho jata hai. Isliye har epoch ka checkpoint `/kaggle/working/runs/aegis_killfeed/weights/last.pt` me save hota rehta hai.
- **Beech me ruka to:** Wahi training cell dobara chalao lekin `resume=True` lagakar — training `last.pt` se aage badhegi, zero se nahi.
- **GPU quota:** Hafte me **~30 ghante** free GPU milta hai. Badi training se pehle Kaggle → Settings me bacha quota dekh lo.
- **Epochs kam karne hon to:** `epochs=50` ko 25-30 kar sakte ho — lekin **default 50 hi rakha hai**.


In [ ]:
# Cell 5 — Training shuru karo
model.train(
    data="/kaggle/working/aegis_ds/dataset.yaml",
    epochs=50,  # kam time ho to 25-30 kar sakte ho (default 50)
    imgsz=640,
    batch=16,
    device=0,  # GPU #0 istemal karo
    project="/kaggle/working/runs",
    name="aegis_killfeed",
    exist_ok=True,
    patience=15,  # 15 epochs tak sudhaar na ho to pehle rok do (time bachega)
    # Beech me ruka tha? Upar wali jagah ye line lagao: resume=True,
)
print("Training khatm ✔")


## Step 5 — Result numbers ka matlab samjho 📊

**Do tarah ke numbers aayenge — farak samajhna zaroori hai:**

1. **"Training ke dauraan" wale numbers (valid set par):** Training ke baad model valid images par kaisa chala — *practice test* jaise.
2. **"Final report card" wale numbers (test set par):** Sabse aakhir me, best model ko **500-1000 clean frames** par parkha jayega jo **ek bilkul alag tournament** se hain aur training me **kabhi use nahi hue**. Ye **"asli duniya ka andaza"** hai — **isi number par bharosa karo.**

Numbers ka matlab (dono jagah same):
- **Precision**: Model ne jitne boxes banaye, unme se kitne sahi the. (Zyada = kam galat alarm)
- **Recall**: Asli kill-feed me se kitne pakde gaye. (Zyada = kam miss)
- **mAP50**: Box 50% overlap par sahi mana jata hai — ye uska overall score hai.
- **mAP50-95**: Aur sakht wala overall score (50% se 95% overlap tak ka average).

Sab 0 se 1 ke beech hote hain; **1 ke jitne kareeb, utna behtar**. Koi number pehle se promise nahi — ye aapke dataset par nirbhar karega.


In [ ]:
# Cell 6 — Validation numbers ("training ke dauraan" wale)
metrics = model.val()  # best weights par valid set check karega
print(f"Precision : {metrics.box.mp:.3f}   (kitne detections sahi the)")
print(f"Recall    : {metrics.box.mr:.3f}   (kitne asli kill-feed pakde gaye)")
print(f"mAP50     : {metrics.box.map50:.3f}  (overall score, 50% overlap)")
print(f"mAP50-95  : {metrics.box.map:.3f}  (sakht wala overall score)")
print("\n(Ye 'training ke dauraan' wale numbers hain — final report card agle cell me.)")


## Step 5b — FINAL report card 📝 (test set: asli duniya ka andaza)

Ab best model ko **test set** par parkho — 500-1000 clean frames jo **ek bilkul alag tournament** se hain aur training me **kabhi use nahi hue**. Jo numbers yahan aayenge, wahi **asli duniya ka sabse imaandaar andaza** hai.


In [ ]:
# Cell 7 — FINAL evaluation: test set par (final report card / asli duniya ka andaza)
import glob

test_imgs = glob.glob("/kaggle/working/aegis_ds/images/test/*")
if not test_imgs:
    print("⚠️ test/ split nahi mila — final report card skip ho raha hai.")
    print("   Dataset me 'test' folder me 500-1000 clean frames (ek alag tournament se) jodho.")
else:
    print(f"test/ me {len(test_imgs)} clean frames — best model ko parakh rahe hain...")
    tm = model.val(split="test")  # dataset.yaml ke 'test:' path par chalega
    print("\n===== FINAL REPORT CARD (test set: alag tournament ke clean frames) =====")
    print(f"Precision : {tm.box.mp:.3f}")
    print(f"Recall    : {tm.box.mr:.3f}")
    print(f"mAP50     : {tm.box.map50:.3f}")
    print(f"mAP50-95  : {tm.box.map:.3f}")
    print("\nYe 'asli duniya ka andaza' hai — ye frames training me kabhi use nahi hue.")
    print("Valid-set wale numbers 'training ke dauraan' the; YE final report card hai.")


## Step 6 — ONNX export 📦

Ye ONNX **seedha** AEGIS ke `VISION_MODEL_PATH` me lagta hai — koi extra post-processing nahi chahiye.


In [ ]:
# Cell 8 — best.pt → best.onnx
import shutil, os, glob

onnx_path = model.export(format="onnx")  # runs/aegis_killfeed/weights/best.onnx banega
shutil.copy(str(onnx_path), "/kaggle/working/best.onnx")
print(f"✔ /kaggle/working/best.onnx taiyaar ({os.path.getsize('/kaggle/working/best.onnx')/1e6:.1f} MB)")


## ✅ Ho gaya! Ab `best.onnx` download karo ⬇️

1. Right side **Output** panel se `best.onnx` download karo.
2. Server par `VISION_MODEL_PATH=/path/to/best.onnx` set karo, backend restart karo (poora tarika `docs/KAGGLE_TRAINING.md` me).
3. RF-DETR wale notebook ka result isse compare karo — jo behtar ho, wahi final rakho.


In [ ]:
# Cell 9 — Final check
import os
p = "/kaggle/working/best.onnx"
assert os.path.exists(p), "best.onnx nahi mila — Cell 7 dobara chalao!"
print(f"✔ {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("Ab right side Output panel se download karo ⬇️")
